# Data Loading 

In [1]:
import pandas as pd
import joblib
import os

df = pd.read_csv('../datasets/processed/Cleaned_Dataset_validated.csv')
df_sampled = df.sample(frac=0.3, random_state=42)
X = df_sampled.drop(columns=['diseases'])
y = df_sampled['diseases']

# Processing

In [2]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Different ML Agorithms

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB()
}

best_model = None
best_accuracy = 0
best_model_name = ""

# Model Training, Evaluate Metrics & Comparison 

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("--- Training and Evaluating ML Models for Disease Prediction ---\n")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f"Model: {name}")
    print(f"  -> Accuracy  : {acc:.4f}")
    print(f"  -> Precision : {prec:.4f}")
    print(f"  -> Recall    : {rec:.4f}")
    print(f"  -> F1-Score  : {f1:.4f}\n")
    
    # Best model ko track karna
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

print(f"=== Best Model Selected: {best_model_name} (Accuracy: {best_accuracy:.4f}) ===")

--- Training and Evaluating ML Models for Disease Prediction ---

Model: Random Forest
  -> Accuracy  : 0.7994
  -> Precision : 0.8059
  -> Recall    : 0.7994
  -> F1-Score  : 0.7980

Model: Logistic Regression
  -> Accuracy  : 0.8215
  -> Precision : 0.8230
  -> Recall    : 0.8215
  -> F1-Score  : 0.8161

Model: Decision Tree
  -> Accuracy  : 0.7252
  -> Precision : 0.7376
  -> Recall    : 0.7252
  -> F1-Score  : 0.7254

Model: K-Nearest Neighbors (KNN)
  -> Accuracy  : 0.7530
  -> Precision : 0.7722
  -> Recall    : 0.7530
  -> F1-Score  : 0.7546

Model: Naive Bayes
  -> Accuracy  : 0.8387
  -> Precision : 0.8783
  -> Recall    : 0.8387
  -> F1-Score  : 0.8464

=== Best Model Selected: Naive Bayes (Accuracy: 0.8387) ===


# Model Persistence

In [9]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(encoder, '../models/disease_encoder.pkl')

print("Best model ('best_model.pkl') and disease encoder ('disease_encoder.pkl') successfully saved '../models/' folder!")

Best model ('best_model.pkl') and disease encoder ('disease_encoder.pkl') successfully saved '../models/' folder!


In [ ]:
import numpy as np

def predict_disease(symptoms_list, feature_columns):
    # Saare symptoms ko 0 ya 1 ki list mein convert karna jo model expect karta hai
    input_data = [1 if col in symptoms_list else 0 for col in feature_columns]
    input_array = np.array(input_data).reshape(1, -1)
    
    # Prediction karna
    pred_encoded = best_model.predict(input_array)
    predicted_disease = encoder.inverse_transform(pred_encoded)
    
    return predicted_disease[0]



In [11]:
# Apne dataset ke columns ke mutabiq kuch symptoms ki list dein
# (Dhyan rahe ki yeh wahi naam hone chahiye jo aapke dataset ke columns mein hain)
sample_symptoms = ['anxiety', 'depressio', 'shortness_of_breath']

# Function ko call karna
result = predict_disease(sample_symptoms, X.columns)
print("Predicted Disease:", result)

Predicted Disease: central atherosclerosis


c:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(
